In [10]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import math
import matplotlib.dates as mdates
from sklearn import datasets, linear_model
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.lines import Line2D
import statsmodels.api as sm
from scipy.stats import t
import random

import os
import pickle
import cvxpy as cp
from tqdm import tqdm
import seaborn as sns
import torch
import copy

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import openpyxl

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torch.autograd import Function

from sklearn.preprocessing import StandardScaler

from functions import CVaRSolver, SPOPlus, VARasNN
import functions

In [2]:
data_path = "../data/processed/"

return_data = pd.read_csv(data_path + "stationary_return_data.csv")

In [3]:
# Pivot return data
return_matrix = return_data.pivot(index='date', columns='RIC', values='return')
display(return_matrix)

RIC,A.N,AAPL.OQ,ABT.N,ACGL.OQ,ADBE.OQ,ADI.OQ,ADM.N,ADP.OQ,ADSK.OQ,AEE.N,...,WMB.N,WMT.N,WST.N,WY.N,XEL.OQ,XOM.N,XRAY.OQ,YUM.N,ZBRA.OQ,ZION.OQ
date,,,,,,,,,,,,,,,,,,,,,
2000-01-31,-0.143897,0.009119,-0.097065,0.158416,-0.181227,0.005376,-0.035897,-0.119490,-0.092685,-5.725191e-03,...,0.267894,-0.207957,0.005608,-0.195522,-0.012821,0.036462,0.047619,-0.258900,0.011752,0.003960
2000-02-29,0.566572,0.104819,0.003831,0.042735,0.852440,0.679144,-0.139973,-0.081686,0.462168,-7.869482e-02,...,0.079032,-0.110731,-0.048485,-0.105664,-0.087662,-0.092836,0.035354,-0.069869,0.124604,-0.102537
2000-03-31,0.003014,0.184842,0.074427,0.073770,0.091363,0.026274,0.031056,0.109845,0.018182,5.474215e-02,...,0.054239,0.141254,-0.140127,0.110840,0.131673,0.033195,0.109834,0.166667,-0.248826,-0.215548
2000-04-28,-0.147837,-0.086516,0.097600,-0.060115,0.086468,-0.046548,-0.042169,0.115285,-0.155477,1.858586e-01,...,-0.150782,-0.002252,-0.027729,-0.062500,0.116715,-0.001606,0.024229,0.098592,0.140000,-0.003003
2000-05-31,-0.169252,-0.322922,0.058537,-0.025381,-0.069251,0.002441,0.207154,0.020906,-0.030945,-7.407630e-12,...,0.113903,0.040632,-0.038363,-0.064345,0.014327,0.078095,0.055914,-0.141026,-0.157895,0.131392
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-09-29,-0.076402,-0.088678,-0.058795,0.037080,-0.088390,-0.032262,-0.048928,-0.050382,-0.067721,-4.845823e-02,...,-0.011417,-0.016481,-0.077882,-0.063817,0.010451,0.057469,-0.075219,-0.034318,-0.139922,-0.017183
2023-10-31,-0.073692,-0.002570,-0.018228,0.087442,0.043460,-0.101434,-0.051047,-0.092942,-0.044850,1.175999e-02,...,0.021075,0.021760,-0.151702,-0.064253,0.035827,-0.099762,-0.109778,-0.032656,-0.114573,-0.115792
2023-11-30,0.236335,0.113747,0.103014,-0.034495,0.148386,0.165576,0.036457,0.053616,0.105247,2.483159e-02,...,0.069477,-0.047243,0.102681,0.099338,0.026489,-0.020540,0.044064,0.043727,0.131548,0.168970


## Experimental Pipeline

For each of 5 randomly sampled 15-asset subsets of the S&P 500:

For each test month (rolling walk-forward evaluation over the last 12 months of the sample):

1. **Data preparation**: Split data into training, validation (last 6 months before test month), and test (single month) sets. Features $z_{t-1}$ consist of the last 3 months of returns (VAR(3) lag structure).

2. **Scenario matrix**: Bootstrap 1000 return scenarios from the training period to approximate the empirical return distribution for the CVaR constraint. A separate scenario matrix including the validation period is used for test-month inference.

3. **Oracle solutions**: Precompute the optimal portfolio $w^\star(y_t)$ for every training and validation instance by solving the CVaR-LP with true returns as input. These are required to compute regret during training and early stopping.

4. **Loss normalization**: Compute SPO+ and MSE scale factors on the initialized model to ensure the two loss components are comparable when combined.

5. **Model training**: For each $\gamma \in \{0, 0.25, 0.5, 0.75, 1.0\}$, train a VAR(3) model (implemented as a linear layer) by minimizing the combined loss
$$\mathcal{L}_{\text{combined}} = \gamma \cdot \mathcal{L}_{\text{SPO+}} + (1 - \gamma) \cdot \mathcal{L}_{\text{MSE}}$$
using Adam with learning rate $10^{-3}$, batch size 16, and early stopping based on validation regret with patience 2. Model weights are restored from the epoch achieving the lowest validation regret.

6. **Test inference**: The trained model predicts returns $\hat{y}_{\text{test}}$ for the test month. The CVaR-LP is solved with $\hat{y}_{\text{test}}$ to obtain portfolio weights $\hat{w}$, and again with true returns $y_{\text{test}}$ to obtain the oracle portfolio $w^\star$. Realized return, oracle return, and regret are recorded.

7. **Checkpointing**: Results are saved to disk after every (subset, test month) combination to prevent data loss.

Results are aggregated across all test months and asset subsets to compare DFL ($\gamma > 0$) against pure MSE training ($\gamma = 0$) in terms of realized portfolio return, test regret, and prediction accuracy.

## Specify parameters

In [11]:
beta = 0.09                 # CVaR threshold (justification in thesis)

n_test_months = 12          # last year as test period

gamma_levels = [0, 0.25, 0.5, 0.75, 1.0]    # similiar to Lee et al.

n_epochs = 10               # HYPERPARAM: number of epochs
val_length = 6              # HYPERPARAM: validation length
max_lag = 3                 # HYPERPARAM: lags of VAR model

## Final Loop

In [12]:
results = []

for data_seed in range(1,6):
    # prepare random return data subset
    random.seed(data_seed)
    cols_subset = random.sample(list(return_matrix.columns), 15)
    return_matrix_subset = return_matrix[cols_subset]
    # Save preprocessed return data subset
    return_matrix_subset.to_csv(f"../data/processed/return_matrix_random_seed{data_seed}.csv", index=False)
    
    X, Y = functions.create_time_series_data_with_lags(return_matrix_subset, max_lag)

    
    for test_index in tqdm(range(1,n_test_months+1), desc="test months"):

        print(f"\nDataset {data_seed}; Test Index = {test_index}\n")
        print("Preparing Data...")

        # data preparation
        X_train, X_val, X_test, Y_train, Y_val, Y_test = functions.create_train_test_split(X, Y, test_index, val_length)
        train_scenario_loss_matrix = functions.get_scenario_loss_matrix(return_matrix_subset, test_index + val_length, num_scenarios=1000, random_seed=42)
        S, N = train_scenario_loss_matrix.shape # S scenarios × N assets

        # solver
        train_solver = CVaRSolver(
            loss_matrix=train_scenario_loss_matrix,
            N=N,
            S=S,
            alpha=0.95, # CVaR confidence level
            beta=beta
        )

        # Precompute oracle solutions
        oracle_solutions = [
            train_solver.solve(c = - mu).copy()
            for mu in tqdm(Y_train, desc="Computing oracle solutions")
        ]
        oracle_tensor = torch.tensor(
            np.array(oracle_solutions),
            dtype=torch.float32
        )

        # Precompute oracle solutions for validation set
        oracle_val_solutions = [
            train_solver.solve(c=-mu).copy()
            for mu in Y_val
        ]
        oracle_val_tensor = torch.tensor(
            np.array(oracle_val_solutions),
            dtype=torch.float32
        )


        # Prepare dataset for pytorch training
        x_scaler = StandardScaler()
        X_train = x_scaler.fit_transform(X_train)
        X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
        Y_train_tensor = torch.tensor(Y_train, dtype=torch.float32)
        train_dataset = TensorDataset(
            X_train_tensor,
            Y_train_tensor,
            oracle_tensor
        )
        batch_size = 16                                                          # HYPERPARAM: Batch size
        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,        # TensorDataset shuffles all tensors together
            drop_last=False
        )

        # scale validation data using the training scaler
        X_val_scaled = x_scaler.transform(X_val)
        X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
        Y_val_tensor = torch.tensor(Y_val, dtype=torch.float32)

        # specify model for scale computation
        model = VARasNN(input_dim=X_train.shape[1], output_dim=Y_train.shape[1])
        criterion = nn.MSELoss()
        # compute scale factor for the two losses to make them comparable
        spo_scale, mse_scale = functions.compute_loss_normalization(model, train_loader, train_solver, criterion)
        print(f"SPO+ scale: {spo_scale:.6f}", f"; MSE  scale: {mse_scale:.6f}\n")

        print("Train Models for different loss combinations:")
        
        for gamma in gamma_levels:
            
            print(f"\nGamma = {gamma} (weight of SPO+ loss)\n")

            # specify model and optimizer
            torch.manual_seed(42)
            model = VARasNN(input_dim=X_train.shape[1], output_dim=Y_train.shape[1])
            optimizer = torch.optim.Adam(
                model.parameters(),
                lr=1e-3                                                             # HYPERPARAM: Learning rate
            )
            criterion = nn.MSELoss()

            # Train model and retrieve training history
            history = functions.train_model(model,
                                            n_epochs,
                                            train_loader,
                                            optimizer,
                                            criterion,
                                            train_solver,
                                            spo_scale,
                                            mse_scale,
                                            gamma,
                                            X_val_tensor,
                                            Y_val_tensor,
                                            oracle_val_tensor,
                                            early_stopping_patience=2
                                            )

            # Inference
            
            # Scale test data using training scaler
            X_test_scaled = x_scaler.transform(X_test.reshape(1, -1))

            X_test_tensor = torch.tensor(
                X_test_scaled,
                dtype=torch.float32
            )

            # Predict expected returns
            model.eval()
            with torch.no_grad():
                mu_hat_test = model(X_test_tensor)

            mu_hat_test = mu_hat_test.cpu().numpy()[0]

            test_mse = np.mean((mu_hat_test - Y_test)**2)

            # Compute portfolio weights from predicted returns with test solver
            test_scenario_loss_matrix = functions.get_scenario_loss_matrix(return_matrix_subset, test_index, num_scenarios=1000, random_seed=42)
            # test solver
            test_solver = CVaRSolver(
                loss_matrix=test_scenario_loss_matrix,
                N=N,
                S=S,
                alpha=0.95, # CVaR confidence level
                beta=beta
            )

            # Compute portfolio weights using test solver (train + val scenario matrix)
            w_hat = test_solver.solve(c=-mu_hat_test).copy()
            w_oracle = test_solver.solve(c=-Y_test).copy()

            # Realized returns
            realized_return = float(Y_test @ w_hat)
            oracle_return   = float(Y_test @ w_oracle)

            # True regret: how much return we lost by using predicted rather than true returns
            test_regret = oracle_return - realized_return

            results.append({

                # id
                "run_id": f"seed{data_seed}_g{gamma}_t{test_index}_b{beta}",

                # experiment settings
                "data_seed": data_seed,
                "beta": beta,
                "gamma": gamma,
                "test_index": test_index,

                # dimensions
                "n_train": len(X_train),
                "n_assets": N,
                "n_scenarios": S,

                # normalization factors
                "spo_scale": spo_scale,
                "mse_scale": mse_scale,

                # forecasting results
                "Y_hat_test": mu_hat_test,
                "Y_test": Y_test,
                "test_mse": test_mse,

                # portfolio results
                "weights": w_hat,
                "w_oracle": w_oracle,
                "realized_return": realized_return,
                "oracle_return": oracle_return,
                "test_regret": test_regret,

                # training history
                "history": history,
                "final_spo_loss": history["spo_loss"][-1],
                "final_mse_loss": history["mse_loss"][-1],
                "final_combined_loss": history["combined_loss"][-1],
                "early_stopping_epoch": history["early_stopping_epoch"],
                "best_epoch": history["best_epoch"],
                "best_val_regret": min(history["val_regret"])
            })

        print("\n---------------------------------------------------")

        # save results after every combination of data_seed and test_index
        with open("results_checkpoint_tmp.pkl", "wb") as f:
            pickle.dump(results, f)
        os.replace("results_checkpoint_tmp.pkl", "results_checkpoint.pkl")

test months:   0%|          | 0/12 [00:00<?, ?it/s]


Dataset 1; Test Index = 1

Preparing Data...


Computing loss scales for normalization: 100%|██████████| 18/18 [00:31<00:00,  1.76s/it]


SPO+ scale: 0.986078 ; MSE  scale: 0.354258

Train Models for different loss combinations:

Gamma = 0 (weight of SPO+ loss)

Epoch 1: combined=0.883654 | SPO+ scaled=0.928057 | MSE scaled=0.883654 | val_regret=0.034198  ✓ regret improved
Epoch 2: combined=0.651526 | SPO+ scaled=0.809694 | MSE scaled=0.651526 | val_regret=0.031970  ✓ regret improved
Epoch 3: combined=0.507541 | SPO+ scaled=0.708940 | MSE scaled=0.507541 | val_regret=0.034602
Epoch 4: combined=0.410224 | SPO+ scaled=0.644859 | MSE scaled=0.410224 | val_regret=0.036639
Early stopping at epoch 4. Best val regret: 0.031970

Gamma = 0.25 (weight of SPO+ loss)

Epoch 1: combined=0.895022 | SPO+ scaled=0.926657 | MSE scaled=0.884477 | val_regret=0.034236  ✓ regret improved
Epoch 2: combined=0.689712 | SPO+ scaled=0.800928 | MSE scaled=0.652640 | val_regret=0.032005  ✓ regret improved
Epoch 3: combined=0.556392 | SPO+ scaled=0.697350 | MSE scaled=0.509406 | val_regret=0.035543
Epoch 4: combined=0.467569 | SPO+ scaled=0.630778 |

test months:   8%|▊         | 1/12 [10:29<1:55:26, 629.72s/it]

...

SPO+ scale: 0.930575 ; MSE  scale: 0.338774

Train Models for different loss combinations:

Gamma = 0 (weight of SPO+ loss)

Epoch 1: combined=0.947672 | SPO+ scaled=1.022194 | MSE scaled=0.947672 | val_regret=0.040996  ✓ regret improved
Epoch 2: combined=0.686292 | SPO+ scaled=0.851940 | MSE scaled=0.686292 | val_regret=0.041840
Epoch 3: combined=0.525492 | SPO+ scaled=0.758766 | MSE scaled=0.525492 | val_regret=0.042610
Early stopping at epoch 3. Best val regret: 0.040996

Gamma = 0.25 (weight of SPO+ loss)

Epoch 1: combined=0.966923 | SPO+ scaled=1.021256 | MSE scaled=0.948812 | val_regret=0.041029  ✓ regret improved
Epoch 2: combined=0.728318 | SPO+ scaled=0.844635 | MSE scaled=0.689545 | val_regret=0.042018
Epoch 3: combined=0.583579 | SPO+ scaled=0.747950 | MSE scaled=0.528788 | val_regret=0.042134
Early stopping at epoch 3. Best val regret: 0.041029

Gamma = 0.5 (weight of SPO+ loss)

Epoch 1: combined=0.989356 | SPO+ scaled=1.023019 | MSE scaled=0.955693 | val_regret=0.041125 

test months:  50%|█████     | 6/12 [1:03:48<59:37, 596.19s/it]

Epoch 4: combined=0.710569 | SPO+ scaled=0.710569 | MSE scaled=0.607181 | val_regret=0.044276
Early stopping at epoch 4. Best val regret: 0.040193

---------------------------------------------------

Dataset 1; Test Index = 7

Preparing Data...


test months:  50%|█████     | 6/12 [1:04:36<1:04:36, 646.01s/it]


KeyboardInterrupt: 

In [6]:
results

[{'run_id': 'seed1_g0_t1_b0.09',
  'data_seed': 1,
  'beta': 0.09,
  'gamma': 0,
  'test_index': 1,
  'n_train': 279,
  'n_assets': 15,
  'n_scenarios': 1000,
  'spo_scale': np.float64(0.896789320301087),
  'mse_scale': np.float64(0.32578633891211617),
  'Y_hat_test': array([-0.5241198 ,  0.39202166, -0.4682587 , -0.28422308,  0.41093853,
          0.23335159, -0.38380623, -0.3404418 ,  0.26588136, -0.79438627,
          0.2899094 ,  0.17572895, -1.0548847 , -0.27633828,  0.67496496],
        dtype=float32),
  'Y_test': array([ 0.06263721, -0.0040937 , -0.04385215,  0.06797651,  0.02032161,
         -0.0495874 , -0.00514266, -0.01352265,  0.03680063, -0.01431667,
         -0.00076948, -0.01120596,  0.1246969 ,  0.03237282,  0.05350714]),
  'test_mse': np.float64(0.2627736924322282),
  'weights': array([-0.        ,  0.2       , -0.        , -0.        ,  0.2       ,
         -0.        , -0.        , -0.        ,  0.07197462, -0.        ,
          0.2       , -0.        , -0.        ,

In [7]:
with open("results_checkpoint.pkl", "rb") as f:
    results = pickle.load(f)

In [8]:
results[0]

{'run_id': 'seed1_g0_t1_b0.09',
 'data_seed': 1,
 'beta': 0.09,
 'gamma': 0,
 'test_index': 1,
 'n_train': 279,
 'n_assets': 15,
 'n_scenarios': 1000,
 'spo_scale': np.float64(0.896789320301087),
 'mse_scale': np.float64(0.32578633891211617),
 'Y_hat_test': array([-0.5241198 ,  0.39202166, -0.4682587 , -0.28422308,  0.41093853,
         0.23335159, -0.38380623, -0.3404418 ,  0.26588136, -0.79438627,
         0.2899094 ,  0.17572895, -1.0548847 , -0.27633828,  0.67496496],
       dtype=float32),
 'Y_test': array([ 0.06263721, -0.0040937 , -0.04385215,  0.06797651,  0.02032161,
        -0.0495874 , -0.00514266, -0.01352265,  0.03680063, -0.01431667,
        -0.00076948, -0.01120596,  0.1246969 ,  0.03237282,  0.05350714]),
 'test_mse': np.float64(0.2627736924322282),
 'weights': array([-0.        ,  0.2       , -0.        , -0.        ,  0.2       ,
        -0.        , -0.        , -0.        ,  0.07197462, -0.        ,
         0.2       , -0.        , -0.        ,  0.14456499,  0.1834

In [13]:
results

[{'run_id': 'seed1_g0_t1_b0.09',
  'data_seed': 1,
  'beta': 0.09,
  'gamma': 0,
  'test_index': 1,
  'n_train': 279,
  'n_assets': 15,
  'n_scenarios': 1000,
  'spo_scale': np.float64(0.9860780131190068),
  'mse_scale': np.float64(0.35425762914948994),
  'Y_hat_test': array([-0.4066921 ,  0.44550806, -0.25962782, -0.29390946,  0.4328537 ,
          0.3196557 , -0.2842099 , -0.21525414,  0.1518494 , -0.8064904 ,
          0.20778704,  0.19040465, -0.8616893 , -0.15610644,  0.5398214 ],
        dtype=float32),
  'Y_test': array([ 0.06263721, -0.0040937 , -0.04385215,  0.06797651,  0.02032161,
         -0.0495874 , -0.00514266, -0.01352265,  0.03680063, -0.01431667,
         -0.00076948, -0.01120596,  0.1246969 ,  0.03237282,  0.05350714]),
  'test_mse': np.float64(0.19966210234765425),
  'weights': array([-0.        ,  0.2       , -0.        , -0.        ,  0.2       ,
          0.05649723, -0.        , -0.        , -0.        , -0.        ,
          0.2       , -0.        , -0.       